# Evaluación temporal de candidatos — versión 2
Este notebook mejora el proceso de evaluación; no promete mejores predicciones. Usa las mismas 300 lecturas verificadas en Supabase y los mismos cuatro secretos del notebook original. No publica ni reemplaza automáticamente el modelo web.

Compara Random Forest, Extra Trees y regresión logística con umbrales 0.3, 0.5 y 0.7, solo dentro de las 240 filas de desarrollo. Cuatro bloques ordenados, separados por cinco filas, evalúan 160 observaciones. El intervalo de cinco filas no representa una duración constante ni garantiza independencia entre ensayos. La ponderación exploratoria de dos por falso negativo y uno por falso positivo requiere validación operativa.

Las 60 filas finales ya se inspeccionaron anteriormente: sirven para regresión, no como una prueba independiente inédita. Se requieren datos nuevos para aprobar un modelo. No hay datos geográficos para pronosticar incendios por distrito.


In [ ]:
import json, hashlib, platform, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
try:
    from IPython.display import display
except ImportError:
    display = print
OUT = Path('salida_satipo'); OUT.mkdir(exist_ok=True)
print('Python:', platform.python_version(), '| scikit-learn:', sklearn.__version__)


## 1. Conectar Supabase y verificar las 300 filas
En el panel **Secretos** de Colab (icono de llave), agrega y habilita acceso para este notebook:
- `SUPABASE_URL`: URL HTTPS de tu proyecto, sin barra final.
- `SUPABASE_PUBLISHABLE_KEY`: clave pública publishable o anon, nunca secret/service_role.
- `SATIPO_EMAIL`: correo de una cuenta existente del aplicativo.
- `SATIPO_PASSWORD`: contraseña de esa cuenta; no es la contraseña de PostgreSQL.

La consulta utiliza autenticación de usuario y RLS. No crea cuentas ni escribe lecturas. Los secretos no se imprimen ni se exportan. En ejecución local se leen las mismas variables de entorno.
Se comprueban total remoto, 300 IDs únicos, esquema, fechas, etiquetas y hash normalizado de contenido. Si los datos cambiaron, se detiene el entrenamiento para revisar la nueva versión; no actualizar el hash a ciegas.


In [ ]:
"""Consulta de lecturas con sesión de usuario; no realiza escrituras de datos."""
import base64
import hashlib
import json
import math
import re
import urllib.request
import urllib.error
from datetime import datetime, timezone

DATASET = 'smoke-detection-iot-300-v1'
NUMERIC = ['source_row','utc_seconds','temperature_c','humidity_pct','tvoc_ppb','eco2_ppm',
           'raw_h2','raw_ethanol','pressure_hpa','pm1_0','pm2_5','nc0_5','nc1_0','nc2_5','cnt']

def canonical_payload(rows):
    if not isinstance(rows,list) or len(rows)!=300:
        raise ValueError('Se requieren exactamente 300 lecturas; no se entrenará con una respuesta parcial.')
    result=[]
    for row in rows:
        if not isinstance(row,dict) or set(row)!=set(NUMERIC+['dataset_id','recorded_at','fire_alarm']):
            raise ValueError('El esquema de las lecturas no coincide con el esperado.')
        if row['dataset_id']!=DATASET or type(row['fire_alarm']) is not bool:
            raise ValueError('Conjunto de datos o etiqueta inválidos.')
        item={}
        for key in NUMERIC:
            value=row[key]
            if type(value) not in (int,float) or not math.isfinite(value):
                raise ValueError('Hay mediciones no numéricas o no finitas.')
            item[key]=float(value)
        if row['source_row']<0 or row['source_row']!=int(row['source_row']):
            raise ValueError('Identificador de origen inválido.')
        try:
            stamp=datetime.fromisoformat(row['recorded_at'].replace('Z','+00:00'))
        except (AttributeError,TypeError,ValueError):
            raise ValueError('Fecha de lectura inválida.') from None
        if stamp.tzinfo is None or stamp.timestamp()!=row['utc_seconds']:
            raise ValueError('Fecha inconsistente con UTC.')
        item.update(dataset_id=DATASET,recorded_at=stamp.astimezone(timezone.utc).isoformat(),fire_alarm=row['fire_alarm'])
        result.append(item)
    if len({r['source_row'] for r in result})!=300:
        raise ValueError('Se detectaron identificadores duplicados.')
    if sum(r['fire_alarm'] for r in result)!=214:
        raise ValueError('La distribución no coincide con las 214 etiquetas positivas esperadas.')
    return json.dumps(sorted(result,key=lambda r:r['source_row']),sort_keys=True,separators=(',',':'),ensure_ascii=False)

class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        raise RuntimeError('Redirección rechazada; verifica la URL original de Supabase.')

def _request(url, headers, body=None):
    request=urllib.request.Request(url,headers=headers,data=None if body is None else json.dumps(body).encode(),method='GET' if body is None else 'POST')
    try:
        with urllib.request.build_opener(_NoRedirect()).open(request,timeout=40) as response:
            raw=response.read(2_000_001)
            if len(raw)>2_000_000:raise ValueError('Respuesta demasiado grande.')
            return json.loads(raw),response.headers.get('Content-Range','')
    except urllib.error.HTTPError as error:
        status=error.code
        if status in (400,401,403):message='Acceso rechazado: revisa correo, contraseña, clave pública y permisos de lectura.'
        elif status==429:message='Límite de solicitudes alcanzado; espera antes de reintentar.'
        else:message=f'Supabase respondió HTTP {status}. No se entrenó con datos alternativos.'
        raise RuntimeError(message) from None
    except (urllib.error.URLError,TimeoutError):
        raise RuntimeError('No se pudo conectar con Supabase. Revisa conexión y disponibilidad.') from None
    except (json.JSONDecodeError,UnicodeDecodeError):
        raise RuntimeError('Supabase no devolvió JSON válido. No se utilizaron datos alternativos.') from None

def load_supabase_dataset(url, public_key, email, password, expected_sha256):
    if any(not isinstance(value,str) or not value.strip() for value in (url,public_key,email,password)):
        raise ValueError('Los cuatro secretos deben contener texto no vacío.')
    url=url.strip().rstrip('/');public_key=public_key.strip();email=email.strip()
    if not re.fullmatch(r'https://[a-z0-9-]+\.supabase\.co',url):
        raise ValueError('Usa la URL HTTPS del proyecto Supabase, sin rutas ni barra final.')
    role=None
    try:
        encoded=public_key.split('.')[1]
        role=json.loads(base64.urlsafe_b64decode(encoded+'='*(-len(encoded)%4)))['role']
    except (ValueError,IndexError,KeyError,TypeError):pass
    if not public_key.startswith('sb_publishable_') and role!='anon':
        raise ValueError('Solo se admite clave pública publishable o anon; nunca service_role o secret.')
    if not email or not password:raise ValueError('Faltan credenciales de una cuenta existente.')
    session,_=_request(url+'/auth/v1/token?grant_type=password',{'apikey':public_key,'Content-Type':'application/json'}, {'email':email,'password':password})
    token=session.get('access_token') if isinstance(session,dict) else None
    if not isinstance(token,str) or not token:raise RuntimeError('No se obtuvo una sesión válida.')
    rows,content_range=_request(url+'/rest/v1/smoke_readings?select=*&dataset_id=eq.'+DATASET+'&order=source_row.asc&limit=301',
        {'apikey':public_key,'Authorization':'Bearer '+token,'Prefer':'count=exact','Accept':'application/json'})
    if content_range.rsplit('/',1)[-1]!='300':
        raise ValueError('Supabase no confirma un total exacto de 300 filas visibles.')
    payload=canonical_payload(rows)
    if hashlib.sha256(payload.encode()).hexdigest()!=expected_sha256:
        raise ValueError('Las lecturas cambiaron respecto de la muestra aprobada. Revisa los datos antes de entrenar.')
    return rows,payload


In [ ]:
def read_secret(name):
    try:
        from google.colab import userdata
    except ImportError:
        import os
        value = os.environ.get(name)
    else:
        try:
            value = userdata.get(name)
        except Exception:
            raise RuntimeError('Configura el secreto ' + name + ' y autoriza su acceso al notebook.') from None
    if not value:
        raise RuntimeError('Falta el secreto ' + name)
    return value

# Invalida resultados anteriores antes de una nueva consulta, incluso si esta falla.
for previous_name in ('candidate_model','candidate_report','candidate_predictions','rows','payload','df','X','y','X_train','X_test','y_train','y_test',
                      'model','baseline','pred','score','report','predictions','artifact','predecir_medicion'):
    globals().pop(previous_name, None)
rows, payload = load_supabase_dataset(
    read_secret('SUPABASE_URL'), read_secret('SUPABASE_PUBLISHABLE_KEY'),
    read_secret('SATIPO_EMAIL'), read_secret('SATIPO_PASSWORD'),
    expected_sha256='805fc268e338964e489b2e76f39206c4a7b90140d2e9838faafd27c513168a23')
df = pd.DataFrame(rows)
df.to_csv(OUT/'lecturas_300.csv', index=False)
print('Supabase: 300 lecturas verificadas; 214 con alarma y 86 sin alarma.')
print('Hash de la instantánea:', hashlib.sha256(payload.encode()).hexdigest())
display(df.head())


In [ ]:
"""Selección temporal dentro de 240 filas; las 60 publicadas son regresión, no prueba inédita."""
import hashlib
import json
import numpy as np
import pandas as pd
import sklearn
import platform
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score

FEATURES=['temperature_c','humidity_pct','tvoc_ppb','eco2_ppm','raw_h2','raw_ethanol','pressure_hpa','pm1_0','pm2_5','nc0_5','nc1_0','nc2_5']
THRESHOLDS=(0.3,0.5,0.7)
NAMES=('random_forest','extra_trees','logistic')

def estimator(name):
    if name=='random_forest':return make_pipeline(SimpleImputer(),RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=3,class_weight='balanced',random_state=20260923,n_jobs=-1))
    if name=='extra_trees':return make_pipeline(SimpleImputer(),ExtraTreesClassifier(n_estimators=300,max_depth=6,min_samples_leaf=3,class_weight='balanced',random_state=20260923,n_jobs=-1))
    if name=='logistic':return make_pipeline(SimpleImputer(),StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=3000,random_state=20260923))
    raise ValueError('Modelo no permitido')

def metrics(y,p):
    tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel()
    return {'accuracy':float(accuracy_score(y,p)),'recall_alarm':float(recall_score(y,p,zero_division=0)),
            'precision_alarm':float(precision_score(y,p,zero_division=0)),
            'false_negatives':int(fn),'false_positives':int(fp),'cost':int(2*fn+fp),
            'confusion_matrix':[[int(tn),int(fp)],[int(fn),int(tp)]]}

def select_model(development):
    """Recibe SOLO desarrollo. Nunca consulta etiquetas de las 60 filas reservadas."""
    if len(development)!=240:raise ValueError('Se requieren 240 filas de desarrollo')
    development=development.sort_values(['utc_seconds','source_row']).reset_index(drop=True)
    X=development[FEATURES].astype(float);y=development.fire_alarm.astype(int)
    if not np.isfinite(X.to_numpy()).all() or y.nunique()!=2:raise ValueError('Datos de desarrollo inválidos')
    folds=[(0,start-5,start,start+40) for start in (80,120,160,200)]
    candidates=[]
    for name in NAMES:
        actual=[];scores=[]
        for lo,end,start,stop in folds:
            if development.iloc[end-1].utc_seconds>=development.iloc[start].utc_seconds:raise ValueError('Fuga temporal')
            model=estimator(name).fit(X.iloc[lo:end],y.iloc[lo:end])
            scores.extend(model.predict_proba(X.iloc[start:stop])[:,list(model.classes_).index(1)])
            actual.extend(y.iloc[start:stop])
        for threshold in THRESHOLDS:
            candidates.append({'name':name,'threshold':threshold,**metrics(actual,np.asarray(scores)>=threshold)})
    # Criterio fijado: FN cuesta 2, FP cuesta 1; desempate reproducible.
    selected=min(candidates,key=lambda r:(r['cost'],r['false_negatives'],r['name'],r['threshold']))
    model=estimator(selected['name']).fit(X,y)
    return model,selected,{'folds':folds,'gap_rows':5,'validation_rows':160,'validation_alarm_rows':int(development.iloc[80:].fire_alarm.sum()),'validation_no_alarm_rows':int((~development.iloc[80:].fire_alarm).sum()),'candidates':candidates,
        'criterion':'minimize 2*FN+FP; exploratory engineering weight, not a validated operational cost'}

def run_experiment(rows):
    df=pd.DataFrame(rows).sort_values(['utc_seconds','source_row']).reset_index(drop=True)
    if len(df)!=300 or df.source_row.nunique()!=300:raise ValueError('Se requieren las 300 filas únicas')
    if not np.isfinite(df[FEATURES].astype(float).to_numpy()).all():raise ValueError('Mediciones no finitas')
    if not all(type(value) is bool for value in df.fire_alarm.tolist()):raise ValueError('Etiquetas inválidas')
    if df.iloc[239].utc_seconds>=df.iloc[240].utc_seconds:raise ValueError('Solapamiento temporal')
    model,selected,validation=select_model(df.iloc[:240].copy())
    X=df.iloc[240:][FEATURES];y=df.iloc[240:].fire_alarm.astype(int)
    score=model.predict_proba(X)[:,list(model.classes_).index(1)]
    prediction=score>=selected['threshold']
    old=estimator('random_forest').fit(df.iloc[:240][FEATURES],df.iloc[:240].fire_alarm.astype(int))
    reference=metrics(y,old.predict(X));candidate=metrics(y,prediction)
    result={'python_version':platform.python_version(),'sklearn_version':sklearn.__version__,'version':'temporal-selection-v2','selected':selected,'validation':validation,
        'published_baseline_regression':reference,'candidate_regression':candidate,
        'promotion':'NOT_APPROVED_REQUIRES_NEW_INDEPENDENT_DATA',
        'limitation':'The 60 rows were previously inspected. Regression results are not an independent generalization estimate. No district or future-fire prediction.',
        'dataset_sha256':hashlib.sha256(json.dumps(rows,sort_keys=True).encode()).hexdigest()}
    output=df.iloc[240:][['source_row','recorded_at','fire_alarm']].copy()
    output['predicted_fire_alarm']=prediction;output['model_score_alarm_uncalibrated']=score
    output['threshold']=selected['threshold']
    return model,result,output


In [ ]:
candidate_model = candidate_report = candidate_predictions = None
candidate_model, candidate_report, candidate_predictions = run_experiment(rows)
print(json.dumps(candidate_report, indent=2))
output_v2 = Path('salida_satipo_v2'); output_v2.mkdir(exist_ok=True)
(output_v2/'evaluacion.json').write_text(json.dumps(candidate_report,indent=2),encoding='utf-8')
candidate_predictions.to_csv(output_v2/'predicciones_regresion.csv',index=False)
print('Candidato no aprobado. El panel conserva la evaluación publicada.')
